<a href="https://colab.research.google.com/github/catrina-llamas-1/Cats-Repository/blob/main/amenities_matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amenities Matcher: Dataset A ↔ Dataset B

This notebook:
1. Loads Dataset A and Dataset B
2. Matches rows by address (`primary_address` in A vs `property_address` in B)
3. Filters Dataset B's `amenities` against a canonical list
4. Writes matched amenities back into Dataset A's `amenities` column
5. Exports the updated Dataset A (retaining its original format)

## 0. Configuration — Edit This Section

In [1]:
# ── File paths ──────────────────────────────────────────────────────────────
DATASET_A_PATH = "dataset_pi.xlsx"   # Supports .xlsx, .xls, or .csv
DATASET_B_PATH = "dataset_costar.xlsx"   # Supports .xlsx, .xls, or .csv
OUTPUT_PATH    = "dataset_a_updated.xlsx"  # Output will match Dataset A's format

# ── Column names (adjust if your files use different headers) ────────────────
A_ADDRESS_COL   = "Primary address"   # Address column in Dataset A
A_AMENITIES_COL = "Amenities"         # Amenities column in Dataset A

B_ADDRESS_COL   = "Property Address"  # Address column in Dataset B
B_AMENITIES_COL = "Amenities"         # Amenities column in Dataset B

# ── How amenities are stored in Dataset B ───────────────────────────────────
# Set the delimiter that separates multiple amenities in a single cell.
# Common values: ","  "|"  ";"  or None if each row has only one amenity.
B_AMENITIES_DELIMITER = ","

# ── How amenities will be written into Dataset A ─────────────────────────────
OUTPUT_DELIMITER = ", "   # Delimiter used when writing multiple amenities back

# ── Overwrite behaviour ──────────────────────────────────────────────────────
# True  → replace any existing value in A's amenities column on a match
# False → only fill when A's amenities cell is empty / NaN
OVERWRITE_EXISTING = True

## 1. Canonical Amenities List

In [2]:
CANONICAL_AMENITIES = {
    "24 Hour Access",
    "Atrium",
    "Banking",
    "Bicycle Storage",
    "Bio-Tech / Lab Space",
    "Car Charging Station",
    "Conference Center",
    "Controlled Access",
    "Courtyard",
    "Coworking Space",
    "Energy Star Labeled",
    "Fitness Center",
    "Food Service",
    "On Transit",
    "Private Terrace",
    "Property Manager on Site",
    "Restaurant",
    "Roof Terrace",
    "Security System",
    "Shower Facilities",
    "Signage",
    "Storage Space",
    "Tenant Lounge",
    "Waterfront",
}

print(f"Canonical list contains {len(CANONICAL_AMENITIES)} amenities.")

Canonical list contains 24 amenities.


## 2. Helper Functions

In [3]:
import pandas as pd
import os


def load_file(path: str) -> pd.DataFrame:
    """Load a CSV or Excel file into a DataFrame."""
    ext = os.path.splitext(path)[-1].lower()
    if ext == ".csv":
        return pd.read_csv(path, dtype=str)
    elif ext in (".xlsx", ".xls"):
        return pd.read_excel(path, dtype=str)
    else:
        raise ValueError(f"Unsupported file type: {ext}. Use .csv, .xlsx, or .xls.")


def normalise_address(addr) -> str:
    """Lowercase + strip whitespace for fuzzy-free exact matching."""
    if pd.isna(addr):
        return ""
    return str(addr).strip().lower()


def parse_amenities(raw, delimiter: str) -> list[str]:
    """Split a raw amenities string into individual trimmed tokens."""
    if pd.isna(raw) or str(raw).strip() == "":
        return []
    return [a.strip() for a in str(raw).split(delimiter) if a.strip()]


def filter_canonical(amenities: list[str], canonical: set) -> list[str]:
    """Keep only amenities that appear in the canonical list (case-sensitive exact match)."""
    return [a for a in amenities if a in canonical]


print("Helper functions loaded.")

Helper functions loaded.


## 3. Load the Datasets

In [4]:
df_a = load_file(DATASET_A_PATH)
df_b = load_file(DATASET_B_PATH)

print(f"Dataset A: {df_a.shape[0]:,} rows × {df_a.shape[1]} columns")
print(f"Dataset B: {df_b.shape[0]:,} rows × {df_b.shape[1]} columns")
print()
print("Dataset A columns:", df_a.columns.tolist())
print("Dataset B columns:", df_b.columns.tolist())

Dataset A: 157 rows × 10 columns
Dataset B: 1,382 rows × 10 columns

Dataset A columns: ['Avant Property ID', 'Primary address', 'Display address', 'Property name', 'City', 'State', 'Amenities', 'Internal Comments', 'Building Description', 'Update']
Dataset B columns: ['Property Address', 'Property Name', 'Submarket Name', 'Submarket Cluster', 'City', 'Postal Code', 'Region Name', 'Amenities', 'Latitude', 'Longitude']


In [ ]:
# Quick preview
print("── Dataset A (first 3 rows) ──")
display(df_a.head(3))

print("── Dataset B (first 3 rows) ──")
display(df_b.head(3))

## 4. Validate Required Columns

In [5]:
def check_col(df, col, label):
    if col not in df.columns:
        raise KeyError(
            f"Column '{col}' not found in {label}.\n"
            f"Available columns: {df.columns.tolist()}\n"
            f"Update the Configuration section at the top of this notebook."
        )
    print(f"  ✓ {label}: '{col}'")

print("Checking columns …")
check_col(df_a, A_ADDRESS_COL,   "Dataset A")
check_col(df_a, A_AMENITIES_COL, "Dataset A")
check_col(df_b, B_ADDRESS_COL,   "Dataset B")
check_col(df_b, B_AMENITIES_COL, "Dataset B")
print("All required columns present.")

Checking columns …
  ✓ Dataset A: 'Primary address'
  ✓ Dataset A: 'Amenities'
  ✓ Dataset B: 'Property Address'
  ✓ Dataset B: 'Amenities'
All required columns present.


## 5. Build Address → Amenities Lookup from Dataset B

In [6]:
# Build a dict: normalised_address → list of canonical amenities
# If multiple Dataset B rows share the same address, their amenities are merged.

b_lookup: dict[str, list[str]] = {}

for _, row in df_b.iterrows():
    addr_key = normalise_address(row[B_ADDRESS_COL])
    if not addr_key:
        continue  # skip blank addresses

    raw_amenities = parse_amenities(row[B_AMENITIES_COL], B_AMENITIES_DELIMITER)
    matched       = filter_canonical(raw_amenities, CANONICAL_AMENITIES)

    if addr_key not in b_lookup:
        b_lookup[addr_key] = []

    # Merge without duplicates
    for amenity in matched:
        if amenity not in b_lookup[addr_key]:
            b_lookup[addr_key].append(amenity)

print(f"Lookup built: {len(b_lookup):,} unique addresses from Dataset B.")
print(f"Addresses with at least one canonical amenity: "
      f"{sum(1 for v in b_lookup.values() if v):,}")

Lookup built: 1,368 unique addresses from Dataset B.
Addresses with at least one canonical amenity: 610


## 6. Update Dataset A's Amenities Column

In [7]:
df_out = df_a.copy()

matched_count   = 0
unmatched_count = 0
skipped_count   = 0

for idx, row in df_out.iterrows():
    addr_key = normalise_address(row[A_ADDRESS_COL])

    if addr_key not in b_lookup:
        unmatched_count += 1
        continue

    canonical_amenities = b_lookup[addr_key]
    if not canonical_amenities:
        unmatched_count += 1
        continue

    existing = row[A_AMENITIES_COL]
    has_value = not (pd.isna(existing) or str(existing).strip() == "")

    if has_value and not OVERWRITE_EXISTING:
        skipped_count += 1
        continue

    df_out.at[idx, A_AMENITIES_COL] = OUTPUT_DELIMITER.join(canonical_amenities)
    matched_count += 1

print(f"Results:")
print(f"  Rows updated   : {matched_count:,}")
print(f"  No match in B  : {unmatched_count:,}")
print(f"  Skipped (OVERWRITE_EXISTING=False): {skipped_count:,}")

Results:
  Rows updated   : 3
  No match in B  : 154
  Skipped (OVERWRITE_EXISTING=False): 0


## 7. Review Changes

In [8]:
# Side-by-side comparison of original vs updated amenities for changed rows
changed_mask = df_out[A_AMENITIES_COL] != df_a[A_AMENITIES_COL]

comparison = pd.DataFrame({
    A_ADDRESS_COL       : df_a.loc[changed_mask, A_ADDRESS_COL],
    "amenities_original": df_a.loc[changed_mask, A_AMENITIES_COL],
    "amenities_updated" : df_out.loc[changed_mask, A_AMENITIES_COL],
})

print(f"{len(comparison):,} rows had their amenities updated.")
display(comparison)

87 rows had their amenities updated.


,Primary address,amenities_original,amenities_updated
10,3489 Allan Drive Southwest,NaN,NaN
11,8560 Roper Rd,"Signage, Atrium, Energy Star Labeled, BOMA Cer...","Atrium, Signage"
14,7609 109 Street Northwest,NaN,NaN
18,5400 99 St,NaN,NaN
21,2920 Calgary Trail,NaN,NaN
...,...,...,...
150,5041 Gateway Boulevard Northwest,NaN,NaN
151,10352 68 Avenue Northwest,NaN,NaN
152,250 Southridge Northwest,NaN,NaN
154,3058 106 Street,NaN,NaN


## 8. Export Updated Dataset A

In [9]:
ext = os.path.splitext(OUTPUT_PATH)[-1].lower()

if ext == ".csv":
    df_out.to_csv(OUTPUT_PATH, index=False)
elif ext in (".xlsx", ".xls"):
    df_out.to_excel(OUTPUT_PATH, index=False)
else:
    raise ValueError(f"Unsupported output format: {ext}")

print(f"✅ Saved updated Dataset A to: {OUTPUT_PATH}")
print(f"   Shape: {df_out.shape[0]:,} rows × {df_out.shape[1]} columns")

✅ Saved updated Dataset A to: dataset_a_updated.xlsx
   Shape: 157 rows × 10 columns


## 9. Diagnostic: Unmatched Addresses in Dataset A

In [10]:
# Identify Dataset A addresses that had NO match in Dataset B
unmatched_rows = df_a[
    ~df_a[A_ADDRESS_COL].apply(normalise_address).isin(b_lookup.keys())
]

print(f"{len(unmatched_rows):,} Dataset A addresses not found in Dataset B:")
display(unmatched_rows[[A_ADDRESS_COL]].drop_duplicates())

153 Dataset A addresses not found in Dataset B:


,Primary address
0,28 Avenue Northwest
1,4607 Calgary Trail Northwest
2,1241 Aster Boulevard
3,4212 Gateway Boulevard Northwest
4,4420 Calgary Trail Northwest
...,...
152,250 Southridge Northwest
153,4805 Gateway Boulevard Northwest
154,3058 106 Street
155,7019 104 St SW


## 10. Diagnostic: Non-Canonical Amenities by Entry

In [11]:
# Build a row-level report: for every Dataset B entry, show which of its amenity
# values were NOT in the canonical list so you can trace exactly where bad values live.

non_canonical_records = []

for _, row in df_b.iterrows():
    all_values   = parse_amenities(row[B_AMENITIES_COL], B_AMENITIES_DELIMITER)
    bad_values   = [a for a in all_values if a not in CANONICAL_AMENITIES]
    good_values  = [a for a in all_values if a in CANONICAL_AMENITIES]

    if bad_values:
        non_canonical_records.append({
            B_ADDRESS_COL            : row[B_ADDRESS_COL],
            "raw_amenities"          : row[B_AMENITIES_COL],
            "non_canonical_values"   : OUTPUT_DELIMITER.join(bad_values),
            "non_canonical_count"    : len(bad_values),
            "canonical_values_kept"  : OUTPUT_DELIMITER.join(good_values) if good_values else "(none)",
        })

df_non_canonical = pd.DataFrame(non_canonical_records)

print(f"{len(df_non_canonical):,} Dataset B entries contained at least one non-canonical amenity value.")

if not df_non_canonical.empty:
    display(df_non_canonical.reset_index(drop=True))
else:
    print("All amenity values across Dataset B matched the canonical list exactly.")

201 Dataset B entries contained at least one non-canonical amenity value.


,Property Address,raw_amenities,non_canonical_values,non_canonical_count,canonical_values_kept
0,8704 51st Ave NW,"Air Conditioning, Signage",Air Conditioning,1,Signage
1,131 1st Ave,"Air Conditioning, Central Heating, Wi-Fi","Air Conditioning, Central Heating, Wi-Fi",3,(none)
2,9452 51st Ave,"Reception, Security System, Signage",Reception,1,"Security System, Signage"
3,10335 95th St NW,"24 Hour Access, Courtyard, Fenced Lot, Securit...",Fenced Lot,1,"24 Hour Access, Courtyard, Security System, St..."
4,10130 103 St NW,"24 Hour Access, Bicycle Storage, Conferencing ...",Conferencing Facility,1,"24 Hour Access, Bicycle Storage, Fitness Center"
...,...,...,...,...,...
196,6616 Yellowhead Trl NW,Air Conditioning,Air Conditioning,1,(none)
197,10214 100 Av,"Air Conditioning, Security System, Signage, Wh...","Air Conditioning, Wheelchair Accessible, Wi-Fi",3,"Security System, Signage"
198,12530 110 Av NW,"Courtyard, Natural Light, Yard","Natural Light, Yard",2,Courtyard
199,4817 Hankin St,"Air Conditioning, Signage, Wheelchair Accessib...","Air Conditioning, Wheelchair Accessible, Wi-Fi...",4,Signage


In [12]:
# Download the entry-level breakdown table as an Excel file
NON_CANONICAL_OUTPUT = "non_canonical_amenities.xlsx"

if not df_non_canonical.empty:
    df_non_canonical.reset_index(drop=True).to_excel(NON_CANONICAL_OUTPUT, index=False)
    print(f"✅ Saved to: {NON_CANONICAL_OUTPUT}")
else:
    print("Nothing to export — no non-canonical amenities were found.")

✅ Saved to: non_canonical_amenities.xlsx


In [13]:
# Summary: unique non-canonical values across all entries, with a count of
# how many entries each one appears in — useful for prioritising fixes.

from collections import Counter

all_bad: list[str] = []
for _, row in df_b.iterrows():
    all_values = parse_amenities(row[B_AMENITIES_COL], B_AMENITIES_DELIMITER)
    all_bad.extend(a for a in all_values if a not in CANONICAL_AMENITIES)

freq = Counter(all_bad)

df_freq = (
    pd.DataFrame(freq.items(), columns=["non_canonical_value", "entry_count"])
    .sort_values("entry_count", ascending=False)
    .reset_index(drop=True)
)

print(f"{len(df_freq):,} distinct non-canonical value(s) found:")
display(df_freq)

34 distinct non-canonical value(s) found:


,non_canonical_value,entry_count
0,Air Conditioning,105
1,Natural Light,37
2,Reception,34
3,Wheelchair Accessible,20
4,Central Heating,18
5,Day Care,15
6,Food Court,14
7,Fenced Lot,14
8,Balcony,14
9,Conferencing Facility,12
